# Step 3 — Augmentation Manifest + 5-Fold CV Assignment

For each train-val image, create 6 rows (original + 5 augmentations: 3 rotations + 2 flips).
Augmentations are stored as metadata — actual transforms applied on-the-fly in the dataloader.

Then assign 5-fold CV splits at the slide level so no slide appears in both train and validation within a fold.

### Load train-val images from step 2

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold

DATA_DIR = Path("data")

images = pd.read_csv(DATA_DIR / "image_catalog.csv")
train_val = images[images["split"] == "train_val"].copy()
test = images[images["split"] == "test"].copy()

print(f"Train-val: {len(train_val)} images")
print(f"Test:      {len(test)} images (untouched)")

### Define augmentation transforms

Each transform is a string tag that the dataloader will interpret at training time:
- `original` — no transform
- `rot90`, `rot180`, `rot270` — rotations
- `flip_h`, `flip_v` — horizontal and vertical flips

This gives a 6× expansion: ~6,800 → ~40,800 training rows.

In [ ]:
AUGMENTATIONS = ["original", "rot90", "rot180", "rot270", "flip_h", "flip_v"]

rows = []
for _, row in train_val.iterrows():
    for aug in AUGMENTATIONS:
        new_row = row.to_dict()
        new_row["augmentation"] = aug
        rows.append(new_row)

train_aug = pd.DataFrame(rows)
print(f"Augmented train-val: {len(train_aug)} rows ({len(train_val)} × {len(AUGMENTATIONS)})")
print(f"\nAugmentation counts:")
print(train_aug["augmentation"].value_counts())

### 5-fold CV at the slide level

Assign each slide to exactly one validation fold using `StratifiedKFold` (stratified by label).
All images from a slide — including all their augmented variants — share the same fold.
This prevents leakage between train and validation within each fold.

In [ ]:
SEED = 42
N_FOLDS = 5

slides_tv = train_val[["slide_id", "label"]].drop_duplicates().reset_index(drop=True)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
slide_to_fold = {}
for fold, (_, val_idx) in enumerate(skf.split(slides_tv, slides_tv["label"])):
    for sid in slides_tv.loc[val_idx, "slide_id"]:
        slide_to_fold[sid] = fold

train_aug["fold"] = train_aug["slide_id"].map(slide_to_fold)

print("Slides per fold:")
fold_slides = slides_tv.copy()
fold_slides["fold"] = fold_slides["slide_id"].map(slide_to_fold)
print(fold_slides.groupby("fold")["label"].value_counts().unstack(fill_value=0))

### Verify fold balance

Check that augmented image counts per fold are roughly equal, and that class ratios are consistent across folds.

In [ ]:
print("Augmented images per fold:")
print(train_aug.groupby("fold").size())

print(f"\nAugmented images per fold × class:")
print(train_aug.groupby(["fold", "label"]).size().unstack(fill_value=0))

print(f"\nFor each fold as validation set:")
for f in range(N_FOLDS):
    val_count = len(train_aug[train_aug["fold"] == f])
    train_count = len(train_aug[train_aug["fold"] != f])
    print(f"  Fold {f}: train={train_count}, val={val_count}")

### Verify no slide leakage within folds

For each fold used as validation, confirm its slides don't appear in the training portion.

In [ ]:
for f in range(N_FOLDS):
    val_slides = set(train_aug[train_aug["fold"] == f]["slide_id"])
    train_slides = set(train_aug[train_aug["fold"] != f]["slide_id"])
    overlap = val_slides & train_slides
    assert len(overlap) == 0, f"LEAKAGE in fold {f}: {overlap}"
print(f"No leakage — all {N_FOLDS} folds clean.")

### Export training manifest

Save two files:
- `train_manifest.csv` — augmented train-val rows with `augmentation` and `fold` columns (used by training loop)
- `test_manifest.csv` — test images, no augmentation (used only for final evaluation)

In [ ]:
test["augmentation"] = "original"
test["fold"] = -1

train_aug.to_csv(DATA_DIR / "train_manifest.csv", index=False)
test.to_csv(DATA_DIR / "test_manifest.csv", index=False)

print(f"Saved {DATA_DIR / 'train_manifest.csv'} — {len(train_aug)} rows")
print(f"Saved {DATA_DIR / 'test_manifest.csv'} — {len(test)} rows")